In [1]:
tabla = 'ctsci10'
col_tabla = 'solcitafec'

In [2]:
import decouple
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys

In [3]:
#CONEXIONES
config = decouple.AutoConfig(' ')
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [4]:
#UPDATE DE FECHA
fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=3", con=connection2)
fecha= fecha.iloc[0, 0]
print(fecha)

now = datetime.now()

query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=3"
c1= text(query)
connection2.execute(c1)

2023-05-01


In [5]:
#CONEXION A DB ORACLE
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname = config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

base2 = pd.read_sql_query(f"SELECT * FROM {tabla} WHERE {col_tabla} >= TO_DATE('{fecha}', 'YYYY-MM-DD')", con=connection0)

connection0.close()

In [6]:
base2.shape

(2033990, 39)

In [7]:
base2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2033990 entries, 0 to 2033989
Data columns (total 39 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   solcitanum              int64         
 1   solcitafec              datetime64[ns]
 2   solcitaoricenasicod     object        
 3   solcitacenasicod        object        
 4   solcitapacsecnum        int64         
 5   solcitaarehoscod        object        
 6   solcitaservhoscod       object        
 7   solcitaactcod           object        
 8   solcitaactespcod        object        
 9   solcitatipocitacod      object        
 10  solcitafecpref          datetime64[ns]
 11  diaprefsolcitcod        object        
 12  turprefsolcitacod       object        
 13  estatensolcitacod       object        
 14  solcitaoricenasioricod  object        
 15  solcitacenasioricod     object        
 16  solcitaactmedorinum     float64       
 17  solcitaoricenasicitcod  object        
 18  so

In [8]:
base2.to_sql(name=f'tmp_{tabla}', con=engine3, if_exists='replace', index=False)

990

In [9]:
query_count_before = f"SELECT COUNT(*) FROM {tabla}"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} antes de la inserción: {cant_antes}")

Cantidad de filas en la tabla ctsci10 antes de la inserción: 10773470


In [10]:
borrando=f"DELETE FROM {tabla} WHERE {col_tabla} >='{fecha}'"
borrado = connection3.execute(borrando)

In [11]:

query = f"""
ALTER TABLE tmp_{tabla}
ALTER COLUMN solcitanum TYPE numeric(10,0),
ALTER COLUMN solcitafec TYPE date,
ALTER COLUMN solcitaoricenasicod TYPE character(1),
ALTER COLUMN solcitacenasicod TYPE character(3),
ALTER COLUMN solcitapacsecnum TYPE numeric(10,0),
ALTER COLUMN solcitaarehoscod TYPE character(2),
ALTER COLUMN solcitaservhoscod TYPE character(3),
ALTER COLUMN solcitaactcod TYPE character(2),
ALTER COLUMN solcitaactespcod TYPE character(3),
ALTER COLUMN solcitatipocitacod TYPE character(1),
ALTER COLUMN solcitafecpref TYPE date,
ALTER COLUMN diaprefsolcitcod TYPE character(1),
ALTER COLUMN turprefsolcitacod TYPE character(1),
ALTER COLUMN estatensolcitacod TYPE character(1),
ALTER COLUMN solcitaoricenasioricod TYPE character(1),
ALTER COLUMN solcitacenasioricod TYPE character(3),
ALTER COLUMN solcitaactmedorinum TYPE numeric(10,0),
ALTER COLUMN solcitaoricenasicitcod TYPE character(1),
ALTER COLUMN solcitacenasicitcod TYPE character(3),
ALTER COLUMN solcitaactmedcitnum TYPE numeric(10,0),
ALTER COLUMN solcitaestregcod TYPE character(1),
ALTER COLUMN solcitausucrecod TYPE character(10),
ALTER COLUMN solcitacrefec TYPE timestamp,
ALTER COLUMN solcitausumodcod TYPE character(10),
ALTER COLUMN solcitamodfec TYPE timestamp,
ALTER COLUMN solcitapricod TYPE numeric(1,0),
ALTER COLUMN solcitaipcrecod TYPE character(15),
ALTER COLUMN solcitausuanucod TYPE character(10),
ALTER COLUMN solcitaipanucod TYPE character(15),
ALTER COLUMN solcitafecanu TYPE timestamp,
ALTER COLUMN solcitaipmodcod TYPE character(15),
ALTER COLUMN solcitamodcrecod TYPE character(1),
ALTER COLUMN solcitamodanucod TYPE character(1),
ALTER COLUMN solcitamodmodcod TYPE character(1),
ALTER COLUMN solcitasecnum TYPE numeric(4,0),
ALTER COLUMN solcitacpscod TYPE character(10),
ALTER COLUMN solcitatramedfissolnum TYPE numeric(10,0),
ALTER COLUMN solcitatiptercod TYPE character(2);

INSERT INTO {tabla} 
(solcitanum,solcitafec,solcitaoricenasicod,solcitacenasicod,solcitapacsecnum,solcitaarehoscod,solcitaservhoscod,solcitaactcod,solcitaactespcod,solcitatipocitacod,solcitafecpref,diaprefsolcitcod,turprefsolcitacod,estatensolcitacod,solcitaoricenasioricod,solcitacenasioricod,solcitaactmedorinum,solcitaoricenasicitcod,solcitacenasicitcod,solcitaactmedcitnum,solcitaestregcod,solcitausucrecod,solcitacrefec,solcitausumodcod,solcitamodfec,solcitapricod,solcitaipcrecod,solcitausuanucod,solcitaipanucod,solcitafecanu,solcitaipmodcod,solcitamodcrecod,solcitamodanucod,solcitamodmodcod,solcitasecnum,solcitacpscod,solcitatramedfissolnum,solcitatiptercod)

SELECT 
solcitanum,solcitafec,solcitaoricenasicod,solcitacenasicod,solcitapacsecnum,solcitaarehoscod,solcitaservhoscod,solcitaactcod,solcitaactespcod,solcitatipocitacod,solcitafecpref,diaprefsolcitcod,turprefsolcitacod,estatensolcitacod,solcitaoricenasioricod,solcitacenasioricod,solcitaactmedorinum,solcitaoricenasicitcod,solcitacenasicitcod,solcitaactmedcitnum,solcitaestregcod,solcitausucrecod,solcitacrefec,solcitausumodcod,solcitamodfec,solcitapricod,solcitaipcrecod,solcitausuanucod,solcitaipanucod,solcitafecanu,solcitaipmodcod,solcitamodcrecod,solcitamodanucod,solcitamodmodcod,solcitasecnum,solcitacpscod,solcitatramedfissolnum,solcitatiptercod


FROM tmp_{tabla};
"""

c1= text(query)
connection3.execute(c1)

In [12]:
#BORRAMOS LAS TABLAS
query2=f"""
SELECT COUNT(*) FROM {tabla} ;
"""
c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla ctsci10 después de la inserción: 10797145
La cantidad de filas insertadas fue de: 23675


In [13]:
query2=f"""
DROP TABLE tmp_{tabla};
"""
c2= text(query2)
cursor=connection3.execute(c2)



In [14]:
connection1.close()
connection2.close()
connection3.close()